Data Preprocessing on Breast Cancer Dataset - 5/21/2026 - aprtay2887

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

df = pd.read_csv(r"C:\Users\ecpi\Documents\breastCancerDataset.csv")

df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 32 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID                  569 non-null    int64  
 1   Diagnosis           569 non-null    object 
 2   radius1             569 non-null    float64
 3   texture1            569 non-null    float64
 4   perimeter1          569 non-null    float64
 5   area1               569 non-null    float64
 6   smoothness1         569 non-null    float64
 7   compactness1        569 non-null    float64
 8   concavity1          569 non-null    float64
 9   concave_points1     569 non-null    float64
 10  symmetry1           569 non-null    float64
 11  fractal_dimension1  569 non-null    float64
 12  radius2             569 non-null    float64
 13  texture2            569 non-null    float64
 14  perimeter2          569 non-null    float64
 15  area2               569 non-null    float64
 16  smoothne

,ID,radius1,texture1,perimeter1,area1,smoothness1,compactness1,concavity1,concave_points1,symmetry1,...,radius3,texture3,perimeter3,area3,smoothness3,compactness3,concavity3,concave_points3,symmetry3,fractal_dimension3
count,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,284.000000,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,...,16.269190,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946
std,164.400426,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,...,4.833242,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061
min,0.000000,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,...,7.930000,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040
25%,142.000000,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,...,13.010000,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460
50%,284.000000,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,...,14.970000,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040
75%,426.000000,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,...,18.790000,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080
max,568.000000,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,...,36.040000,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500


In [2]:
numeric_cols = df.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())
df = df.drop_duplicates()

cols_to_scale = [c for c in ['concavity1', 'concavity3'] if c in df.columns]

if cols_to_scale:
    scaler = StandardScaler()
    df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])

if df['Diagnosis'].dtype == 'object':
    df['Diagnosis'] = df['Diagnosis'].map({'B': 0, 'M': 1})
    
X = df.drop(columns=['Diagnosis'])
Y = df['Diagnosis']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.30, random_state=42)

In [3]:

svm_model = SVC()
svm_model.fit(X_train, Y_train)

Y_pred_svm = svm_model.predict(X_test)


print("SVM Accuracy:", accuracy_score(Y_test, Y_pred_svm))
print(classification_report(Y_test, Y_pred_svm))

SVM Accuracy: 0.935672514619883
              precision    recall  f1-score   support

           0       0.93      0.97      0.95       108
           1       0.95      0.87      0.91        63

    accuracy                           0.94       171
   macro avg       0.94      0.92      0.93       171
weighted avg       0.94      0.94      0.94       171



In [4]:
param_grid_svm = {
    "C": [0.1, 1, 10, 100],
    "kernel": ["linear", "rbf", "poly"]
}

grid_svm = GridSearchCV(SVC(), param_grid_svm, cv=5)
grid_svm.fit(X_train, Y_train)

print("Best SVM Parameters:", grid_svm.best_params_)


Best SVM Parameters: {'C': 1, 'kernel': 'linear'}


In [5]:
best_svm = grid_svm.best_estimator_
Y_pred_svm_tuned = best_svm.predict(X_test)

print("Tuned SVM Accuracy:", accuracy_score(Y_test, Y_pred_svm_tuned))
print(classification_report(Y_test, Y_pred_svm_tuned))

Tuned SVM Accuracy: 0.9649122807017544
              precision    recall  f1-score   support

           0       0.99      0.95      0.97       108
           1       0.93      0.98      0.95        63

    accuracy                           0.96       171
   macro avg       0.96      0.97      0.96       171
weighted avg       0.97      0.96      0.97       171



In [6]:

xgb = XGBClassifier(eval_metric='logloss')
xgb.fit(X_train, Y_train)

Y_pred_xgb = xgb.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(Y_test, Y_pred_xgb))
print(classification_report(Y_test, Y_pred_xgb))

XGBoost Accuracy: 0.9707602339181286
              precision    recall  f1-score   support

           0       0.97      0.98      0.98       108
           1       0.97      0.95      0.96        63

    accuracy                           0.97       171
   macro avg       0.97      0.97      0.97       171
weighted avg       0.97      0.97      0.97       171



In [7]:
param_grid_xgb = {
    "n_estimators": [50, 100],
    "learning_rate": [0.1, 0.2],
    "max_depth": [3, 4]
}

grid_xgb = GridSearchCV(
    XGBClassifier(eval_metric="logloss"),
    param_grid_xgb,
    cv=5
)

grid_xgb.fit(X_train, Y_train)

print("Best XGBoost Parameters:", grid_xgb.best_params_)

best_xgb = grid_xgb.best_estimator_
Y_pred_xgb_tuned = best_xgb.predict(X_test)

print("Tuned XGBoost Accuracy:", accuracy_score(Y_test, Y_pred_xgb_tuned))
print(classification_report(Y_test, Y_pred_xgb_tuned))

Best XGBoost Parameters: {'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 100}
Tuned XGBoost Accuracy: 0.9649122807017544
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       108
           1       0.97      0.94      0.95        63

    accuracy                           0.96       171
   macro avg       0.97      0.96      0.96       171
weighted avg       0.96      0.96      0.96       171



1.) Bias could be implemented into this dataset through sampling bias through rows being more malignant or more benign, then the model learns an unbalanced pattern, demographic bias if the dataset over represents certain ages or patient groups or ethnicities, then predictions might not be generalized, measurement bias through imaging or lab equipment differences possibly skewing radius or texture or perimeter values, and also labeling bias through human error in labeling. This can be mitigated through balancing classes, collecting more diverse data, standardized measurement procedures and audits or cross validation.

2.) The tuned svm performed better as the accuracy increased once tuned whereas for the XGBoost tuning actually decreased in accuracy. This could be due to this being a smaller dataset and linear or semi linearly seperable and typically, SVM performs well under these conditions as opposed to the XGBoost model. It's also important to note that the SVM recalls and precision for B and M was more balanced.